In [1]:
# =========================================================
# [전체 전처리 + target_only 추출 + 이상치 처리 + fill_stage 기록]
# 기존 로직 유지 버전
# =========================================================

import pandas as pd
import numpy as np
import re
import os

# ---------------------------------------------------------
# 0. 경로 설정
# ---------------------------------------------------------
BASE_PATH = "./"  # 필요하면 수정
TRAIN_PATH = os.path.join(BASE_PATH, "train.csv")
WHOLESALE_PATH = os.path.join(BASE_PATH, "TRAIN_전국도매_2018-2021.csv")
SANJI_PATH = os.path.join(BASE_PATH, "TRAIN_산지공판장_2018-2021.csv")

# 저장 파일명
STEP0_PATH = os.path.join(BASE_PATH, "train_remove_pepper_with_stage.csv")
STEP1_PATH = os.path.join(BASE_PATH, "train_step1_with_stage.csv")
STEP2_PATH = os.path.join(BASE_PATH, "train_step2_with_stage.csv")
STEP3_PATH = os.path.join(BASE_PATH, "train_step3_final_with_stage.csv")
TARGET_ONLY_PATH = os.path.join(BASE_PATH, "train_target_only_with_stage.csv")
TARGET_ONLY_CLEANED_PATH = os.path.join(BASE_PATH, "train_target_only_cleaned_with_stage.csv")

# ---------------------------------------------------------
# 1. 데이터 로드
# ---------------------------------------------------------
train = pd.read_csv(TRAIN_PATH)
wholesale = pd.read_csv(WHOLESALE_PATH)
sanji = pd.read_csv(SANJI_PATH)

print("원본 train shape:", train.shape)
print("전국도매 shape:", wholesale.shape)
print("산지공판장 shape:", sanji.shape)

# ---------------------------------------------------------
# 2. 공통 설정
# ---------------------------------------------------------
PRICE_COL = "평균가격(원)"
NORMAL_COL = "평년 평균가격(원)"

TARGET_ITEMS = [
    "건고추", "사과", "감자", "배", "깐마늘(국산)",
    "무", "상추", "배추", "양파", "대파"
]

TARGET_CONDITIONS = [
    ('건고추', '화건', '30 kg', '상품'),
    ('사과', '홍로', '10 개', '상품'),
    ('사과', '후지', '10 개', '상품'),
    ('감자', '감자 수미', '20키로상자', '상'),
    ('배', '신고', '10 개', '상품'),
    ('깐마늘(국산)', '깐마늘(국산)', '20 kg', '상품'),
    ('무', '무', '20키로상자', '상'),
    ('상추', '청', '100 g', '상품'),
    ('배추', '배추', '10키로망대', '상'),
    ('양파', '양파', '1키로', '상'),
    ('대파', '대파(일반)', '1키로단', '상')
]

def extract_weight(unit):
    unit_str = str(unit)
    if '100 g' in unit_str:
        return 0.1
    nums = re.findall(r'\d+\.?\d*', unit_str)
    if nums:
        return float(nums[0])
    return 1.0

def get_weight(row):
    unit = str(row['거래단위'])
    unit_nospace = unit.replace(" ", "")

    if '10개' in unit_nospace:
        if row['품목명'] == '사과':
            return 3.0
        elif row['품목명'] == '배':
            return 5.0

    if '100 g' in unit:
        return 0.1

    nums = re.findall(r'\d+\.?\d*', unit)
    return float(nums[0]) if nums else 1.0

def print_missing_status(df, stage_name):
    print(f"\n--- [{stage_name}] 타겟 품목별 결측치 현황 ---")
    summary = []
    for item in TARGET_ITEMS:
        item_df = df[df['품목명'] == item]
        missing_count = ((item_df[PRICE_COL] <= 0) | (item_df[PRICE_COL].isna())).sum()
        total_count = len(item_df)
        ratio = (missing_count / total_count * 100) if total_count > 0 else 0
        summary.append([item, missing_count, total_count, f"{ratio:.2f}%"])
    summary_df = pd.DataFrame(summary, columns=['품목명', '결측치(0포함)', '전체 행', '결측률'])
    print(summary_df.to_string(index=False))

def apply_winsorization(df, price_col):
    df = df.copy()

    if df[price_col].dtype == 'object':
        df[price_col] = df[price_col].astype(str).str.replace(',', '')
    df[price_col] = pd.to_numeric(df[price_col], errors='coerce')

    def cap_group(group):
        lower_bound = group[price_col].quantile(0.01)
        upper_bound = group[price_col].quantile(0.99)
        group[price_col] = group[price_col].clip(lower=lower_bound, upper=upper_bound)
        return group

    df_cleaned = df.groupby(['품목명', '품종명', '거래단위', '등급'], group_keys=False).apply(cap_group)
    return df_cleaned

# ---------------------------------------------------------
# 3. Step 0: 건고추 노이즈 제거
# ---------------------------------------------------------
df = train.copy()

# fill_stage 초기화
# 0: 원본값
# 1: step1 도매 기반 보간
# 2: step2 추세 기반 보간
# 3: step3 전년/평년 기반 보간
df['fill_stage'] = 0

is_pepper = (df['품목명'] == '건고추')
is_target_pepper = (
    (df['품종명'] == '화건') &
    (df['거래단위'] == '30 kg') &
    (df['등급'] == '상품')
)

noise_pepper_idx = df[is_pepper & ~is_target_pepper].index
df = df.drop(noise_pepper_idx).reset_index(drop=True)

df.to_csv(STEP0_PATH, index=False)

print(f"\n[Step 0 완료] 제거된 건고추 노이즈 행 수: {len(noise_pepper_idx)}")
print(f"저장: {STEP0_PATH}")
print("Step0 shape:", df.shape)

# ---------------------------------------------------------
# 4. Step 1: 전국도매 단가 기반 보간
# - train 값이 0 또는 NaN인 행 중
# - 사과/배 제외
# - 도매 단가(원/kg) * 거래단위 중량
# ---------------------------------------------------------
wholesale_daily = wholesale.groupby(['시점', '품목명', '품종명']).agg({
    '총반입량(kg)': 'sum',
    '총거래금액(원)': 'sum'
}).reset_index()

wholesale_daily['도매_단가_kg'] = np.nan
valid_mask = wholesale_daily['총반입량(kg)'] > 0
wholesale_daily.loc[valid_mask, '도매_단가_kg'] = (
    wholesale_daily.loc[valid_mask, '총거래금액(원)'] /
    wholesale_daily.loc[valid_mask, '총반입량(kg)']
)

df['시점'] = df['시점'].astype(str)
wholesale_daily['시점'] = wholesale_daily['시점'].astype(str)

df = df.merge(
    wholesale_daily[['시점', '품목명', '품종명', '도매_단가_kg']],
    on=['시점', '품목명', '품종명'],
    how='left'
)

df['temp_weight'] = df['거래단위'].apply(extract_weight)

step1_fill_mask = (
    ((df[PRICE_COL] == 0) | (df[PRICE_COL].isna())) &
    (~df['품목명'].isin(['사과', '배'])) &
    (df['도매_단가_kg'].notna())
)

df.loc[step1_fill_mask, PRICE_COL] = (
    df.loc[step1_fill_mask, '도매_단가_kg'] *
    df.loc[step1_fill_mask, 'temp_weight']
)
df.loc[step1_fill_mask, 'fill_stage'] = 1

print(f"\n[Step 1 완료] step1 보간 행 수: {step1_fill_mask.sum()}")

df.drop(columns=['도매_단가_kg', 'temp_weight'], inplace=True)
df.to_csv(STEP1_PATH, index=False)

print(f"저장: {STEP1_PATH}")
print_missing_status(df, "Step 1")

# ---------------------------------------------------------
# 5. Step 2: 최근 가격 추세 보간
# - 도매 전순 -> 산지 전순 -> 도매 전달 -> 산지 전달
# - 사과/배 제외
# ---------------------------------------------------------
w_trend = wholesale.groupby(['시점', '품목명', '품종명']).agg({
    '전순 평균가격(원) PreVious SOON': 'mean',
    '전달 평균가격(원) PreVious MMonth': 'mean'
}).reset_index()

s_trend = sanji.groupby(['시점', '품목명', '품종명']).agg({
    '전순 평균가격(원) PreVious SOON': 'mean',
    '전달 평균가격(원) PreVious MMonth': 'mean'
}).reset_index()

w_trend['시점'] = w_trend['시점'].astype(str)
s_trend['시점'] = s_trend['시점'].astype(str)

df = df.merge(w_trend, on=['시점', '품목명', '품종명'], how='left')
df = df.merge(s_trend, on=['시점', '품목명', '품종명'], how='left', suffixes=('_도매', '_산지'))

df['temp_weight'] = df['거래단위'].apply(extract_weight)

trend_val = df['전순 평균가격(원) PreVious SOON_도매'].fillna(
    df['전순 평균가격(원) PreVious SOON_산지']
).fillna(
    df['전달 평균가격(원) PreVious MMonth_도매']
).fillna(
    df['전달 평균가격(원) PreVious MMonth_산지']
)

step2_fill_mask = (
    ((df[PRICE_COL] == 0) | (df[PRICE_COL].isna())) &
    (~df['품목명'].isin(['사과', '배'])) &
    (trend_val.notna())
)

df.loc[step2_fill_mask, PRICE_COL] = (
    trend_val[step2_fill_mask] *
    df.loc[step2_fill_mask, 'temp_weight']
)
df.loc[step2_fill_mask, 'fill_stage'] = 2

print(f"\n[Step 2 완료] step2 보간 행 수: {step2_fill_mask.sum()}")

drop_cols = [c for c in df.columns if 'PreVious' in c] + ['temp_weight']
df.drop(columns=drop_cols, inplace=True)
df.to_csv(STEP2_PATH, index=False)

print(f"저장: {STEP2_PATH}")
print_missing_status(df, "Step 2")

# ---------------------------------------------------------
# 6. Step 3: 전년/평년 가격 기반 보간
# - 전년 우선, 없으면 평년
# - 사과/배 포함 전체 대상
# ---------------------------------------------------------
reference_year = wholesale.groupby(['시점', '품목명', '품종명']).agg({
    '전년 평균가격(원) PreVious YeaR': 'mean',
    '평년 평균가격(원) Common Year SOON': 'mean'
}).reset_index()

reference_year['시점'] = reference_year['시점'].astype(str)

df = df.merge(reference_year, on=['시점', '품목명', '품종명'], how='left')

df['temp_weight'] = df.apply(get_weight, axis=1)

historical_val = df['전년 평균가격(원) PreVious YeaR'].fillna(
    df['평년 평균가격(원) Common Year SOON']
)

step3_fill_mask = (
    ((df[PRICE_COL] == 0) | (df[PRICE_COL].isna())) &
    (historical_val.notna())
)

df.loc[step3_fill_mask, PRICE_COL] = (
    historical_val[step3_fill_mask] *
    df.loc[step3_fill_mask, 'temp_weight']
)
df.loc[step3_fill_mask, 'fill_stage'] = 3

print(f"\n[Step 3 완료] step3 보간 행 수: {step3_fill_mask.sum()}")

# Step3 이후 0원 제거
before_positive = len(df)
df = df[df[PRICE_COL] > 0].copy()
print(f"0원/음수 제거 후 행 수: {len(df)} (제거 {before_positive - len(df)}행)")

# 기존 로직 유지: 전체 데이터에서 품목별 상하위 1% 컷팅(행 제거)
def clean_outliers_remove(df_in, col):
    def get_cleaned_group(group):
        q1 = group[col].quantile(0.01)
        q3 = group[col].quantile(0.99)
        return group[(group[col] > q1) & (group[col] < q3)]
    return df_in.groupby('품목명', group_keys=False).apply(get_cleaned_group)

before_outlier = len(df)
df = clean_outliers_remove(df, PRICE_COL).copy()
print(f"이상치 제거 후 행 수: {len(df)} (제거 {before_outlier - len(df)}행)")

drop_cols = ['전년 평균가격(원) PreVious YeaR', '평년 평균가격(원) Common Year SOON', 'temp_weight']
df.drop(columns=drop_cols, inplace=True)

# is_imputed 추가
df['is_imputed'] = (df['fill_stage'] != 0).astype(int)

df.to_csv(STEP3_PATH, index=False)

print(f"저장: {STEP3_PATH}")
print_missing_status(df, "Step 3 Final")

print("\nfill_stage 분포 (전체 step3_final)")
print(df['fill_stage'].value_counts(dropna=False).sort_index())

# ---------------------------------------------------------
# 7. 타겟 전용 데이터 추출
# ---------------------------------------------------------
target_dfs = []

for p, v, u, g in TARGET_CONDITIONS:
    mask = (
        (df['품목명'] == p) &
        (df['품종명'] == v) &
        (df['거래단위'] == u) &
        (df['등급'] == g)
    )
    filtered_df = df[mask].copy()
    target_dfs.append(filtered_df)

target_final_df = pd.concat(target_dfs, ignore_index=True)

target_final_df.to_csv(TARGET_ONLY_PATH, index=False)

print(f"\n[타겟 추출 완료]")
print(f"전체 {len(df):,}개 데이터 중 타겟 조건에 맞는 {len(target_final_df):,}개 데이터 추출")
print(f"저장: {TARGET_ONLY_PATH}")

print("\n타겟 데이터 fill_stage 분포")
print(target_final_df['fill_stage'].value_counts(dropna=False).sort_index())

print("\n타겟 품목별 데이터 수")
print(
    target_final_df.groupby(['품목명', '품종명', '거래단위', '등급'])
    .size()
    .reset_index(name='데이터 수')
    .to_string(index=False)
)

# ---------------------------------------------------------
# 8. 타겟 전용 데이터 이상치 처리 (winsorizing)
# - 기존 로직 유지
# - fill_stage / is_imputed는 그대로 보존
# ---------------------------------------------------------
target_cleaned_df = apply_winsorization(target_final_df, PRICE_COL).copy()

# 안전하게 타입 정리
target_cleaned_df[PRICE_COL] = pd.to_numeric(target_cleaned_df[PRICE_COL], errors='coerce')

target_cleaned_df.to_csv(TARGET_ONLY_CLEANED_PATH, index=False)

print(f"\n[target_only_cleaned 저장 완료]")
print(f"저장: {TARGET_ONLY_CLEANED_PATH}")
print("shape:", target_cleaned_df.shape)

print("\ncleaned fill_stage 분포")
print(target_cleaned_df['fill_stage'].value_counts(dropna=False).sort_index())

print("\ncleaned is_imputed 분포")
print(target_cleaned_df['is_imputed'].value_counts(dropna=False).sort_index())

# ---------------------------------------------------------
# 9. 최종 확인
# ---------------------------------------------------------
print("\n--- [최종 상태 보고] ---")
print("NaN 개수:", target_cleaned_df[PRICE_COL].isna().sum())
print("0원 개수:", (target_cleaned_df[PRICE_COL] == 0).sum())

print("\n최종 컬럼")
print(target_cleaned_df.columns.tolist())

print("\n미리보기")
print(target_cleaned_df.head())

원본 train shape: (29376, 7)
전국도매 shape: (176014, 22)
산지공판장 shape: (118628, 21)

[Step 0 완료] 제거된 건고추 노이즈 행 수: 1008
저장: ./train_remove_pepper_with_stage.csv
Step0 shape: (28368, 8)

[Step 1 완료] step1 보간 행 수: 6865
저장: ./train_step1_with_stage.csv

--- [Step 1] 타겟 품목별 결측치 현황 ---
    품목명  결측치(0포함)  전체 행    결측률
    건고추         0   144  0.00%
     사과       399   720 55.42%
     감자      2178  4752 45.83%
      배       262   576 45.49%
깐마늘(국산)         0   288  0.00%
      무        72  4752  1.52%
     상추         0   576  0.00%
     배추       356  3744  9.51%
     양파      1652  9792 16.87%
     대파       375  3024 12.40%

[Step 2 완료] step2 보간 행 수: 80
저장: ./train_step2_with_stage.csv

--- [Step 2] 타겟 품목별 결측치 현황 ---
    품목명  결측치(0포함)  전체 행    결측률
    건고추         0   144  0.00%
     사과       399   720 55.42%
     감자      2178  4752 45.83%
      배       262   576 45.49%
깐마늘(국산)         0   288  0.00%
      무        72  4752  1.52%
     상추         0   576  0.00%
     배추       356  3744  9.51%
     양파   

/tmp/ipykernel_8783/2072668412.py:289: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df_in.groupby('품목명', group_keys=False).apply(get_cleaned_group)



[타겟 추출 완료]
전체 22,726개 데이터 중 타겟 조건에 맞는 1,425개 데이터 추출
저장: ./train_target_only_with_stage.csv

타겟 데이터 fill_stage 분포
fill_stage
0    1425
Name: count, dtype: int64

타겟 품목별 데이터 수
    품목명     품종명   거래단위 등급  데이터 수
     감자   감자 수미 20키로상자  상    140
    건고추      화건  30 kg 상품    140
깐마늘(국산) 깐마늘(국산)  20 kg 상품    141
     대파  대파(일반)   1키로단  상    144
      무       무 20키로상자  상    144
      배      신고   10 개 상품    144
     배추      배추 10키로망대  상    144
     사과      홍로   10 개 상품     19
     사과      후지   10 개 상품    125
     상추       청  100 g 상품    140
     양파      양파    1키로  상    144

[target_only_cleaned 저장 완료]
저장: ./train_target_only_cleaned_with_stage.csv
shape: (1425, 9)

cleaned fill_stage 분포
fill_stage
0    1425
Name: count, dtype: int64

cleaned is_imputed 분포
is_imputed
0    1425
Name: count, dtype: int64

--- [최종 상태 보고] ---
NaN 개수: 0
0원 개수: 0

최종 컬럼
['시점', '품목명', '품종명', '거래단위', '등급', '평년 평균가격(원)', '평균가격(원)', 'fill_stage', 'is_imputed']

미리보기
         시점  품목명 품종명   거래단위  등급     평년 평균가격(원)   평균가격(원)

/tmp/ipykernel_8783/2072668412.py:113: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_cleaned = df.groupby(['품목명', '품종명', '거래단위', '등급'], group_keys=False).apply(cap_group)


In [2]:
step0 = pd.read_csv("train_remove_pepper_with_stage.csv")
step1 = pd.read_csv("train_step1_with_stage.csv")
step2 = pd.read_csv("train_step2_with_stage.csv")
step3 = pd.read_csv("train_step3_final_with_stage.csv")
target = pd.read_csv("train_target_only_with_stage.csv")

target_conditions = [
    ('건고추', '화건', '30 kg', '상품'),
    ('사과', '홍로', '10 개', '상품'),
    ('사과', '후지', '10 개', '상품'),
    ('감자', '감자 수미', '20키로상자', '상'),
    ('배', '신고', '10 개', '상품'),
    ('깐마늘(국산)', '깐마늘(국산)', '20 kg', '상품'),
    ('무', '무', '20키로상자', '상'),
    ('상추', '청', '100 g', '상품'),
    ('배추', '배추', '10키로망대', '상'),
    ('양파', '양파', '1키로', '상'),
    ('대파', '대파(일반)', '1키로단', '상')
]

def filter_target(df):
    out = []
    for p, v, u, g in target_conditions:
        mask = (
            (df['품목명'] == p) &
            (df['품종명'] == v) &
            (df['거래단위'] == u) &
            (df['등급'] == g)
        )
        out.append(df[mask].copy())
    return pd.concat(out, ignore_index=True)

for name, d in [("step0", step0), ("step1", step1), ("step2", step2), ("step3", step3), ("target", target)]:
    td = filter_target(d)
    print(f"\n[{name}]")
    print("행 수:", len(td))
    if 'fill_stage' in td.columns:
        print(td['fill_stage'].value_counts(dropna=False).sort_index())


[step0]
행 수: 1440
fill_stage
0    1440
Name: count, dtype: int64

[step1]
행 수: 1440
fill_stage
0    1440
Name: count, dtype: int64

[step2]
행 수: 1440
fill_stage
0    1440
Name: count, dtype: int64

[step3]
행 수: 1425
fill_stage
0    1425
Name: count, dtype: int64

[target]
행 수: 1425
fill_stage
0    1425
Name: count, dtype: int64


In [3]:
step0_target = filter_target(step0)

missing_mask = (step0_target['평균가격(원)'].isna()) | (step0_target['평균가격(원)'] == 0)

print("step0 타겟 행 수:", len(step0_target))
print("step0 타겟 결측/0 개수:", missing_mask.sum())

display(step0_target[missing_mask].sort_values(['품목명', '품종명', '시점']).head(30))

step0 타겟 행 수: 1440
step0 타겟 결측/0 개수: 0


,시점,품목명,품종명,거래단위,등급,평년 평균가격(원),평균가격(원),fill_stage
